# 04 · Profile Authoring：新增数据域只写 yaml

**目标读者**：领域专家（非引擎开发者）

**核心目的**：证明「零金融词表」不只是口号 —— 新增期货域，只写一个 profile yaml，不改 `engine.py` 一行代码。域知识（主键、同义词、缺失标记、单位、规则清单、交易日历）全部在 yaml 里声明。

**验收标准**：

- 新增 profile 不需要修改 `src/finpass/engine.py`（基线 sha256 不变）；
- `engine.py` 里搜不到任何金融业务词；
- 期货 profile 的复合主键 `合约代码+日期` 真的生效，`limit_up_down` 规则来自 yaml 的通用 `range_check` 声明；
- 用 futures profile 打开 A 股 CSV，**明确拒绝**并点名缺失列，而不是静默降级；
- 用同一个引擎跑两个不同 profile，行为差异全部来自 yaml。

**04 的 src 增量（别被「只加 yaml」遮住）**：

- 新增 `src/finpass/profile.py`：profile 的加载与校验（必填字段、重复 rule id、key 归一、`validate_columns`）；
- 新增 `src/finpass/profiles/futures.yaml` 与 `examples/futures_daily.csv`；
- `session.py` 改为按 profile 取主键 / 日期列 / 词表 / 规则，并按规则 `type` 分发；
- `src/finpass/engine.py` **零改动**（`tests/baselines/engine.sha256` 不变）。

「新域仅加 yaml」指的是**域知识**，加载器本身的工程量单独记在这里，不靠口号含糊过去。

## 大纲

1. 解剖 `profiles/a_stock_daily.yaml`：key / 角色列 / 词表 / 单位 / 规则
2. 新增 `profiles/futures.yaml`：复合主键 + 手数单位 + 涨跌停幅度规则
3. 新增示例数据 `examples/futures_daily.csv`
4. 换 profile 跑通：同一套会话代码，规则命中来自 yaml
5. 负例：缺必需列必须明确拒绝
6. engine 零改动：基线 hash + 无金融词
7. 汇总

In [1]:
import sys
from pathlib import Path

import yaml

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from finpass.profile import load_profile

yaml_path = REPO / "src" / "finpass" / "profiles" / "a_stock_daily.yaml"
raw = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))
print("yaml 字段：", sorted(raw))
assert {"name", "key", "business_rules", "missing_tokens", "units", "columns"} <= set(raw)

a_stock = load_profile("a_stock_daily")
print("key =", a_stock.keys, "| 规则 =", a_stock.rule_ids)
print("缺失标记 =", a_stock.missing_tokens, "| 单位 =", a_stock.units)

yaml 字段： ['business_rules', 'calendar', 'columns', 'date_column', 'key', 'missing_tokens', 'name', 'synonyms', 'units', 'version']
key = ['日期'] | 规则 = ['missing_token', 'duplicate_key', 'ohlc_contradiction', 'unit_ambiguity', 'date_format']
缺失标记 = ['--', '暂无', 'N/A'] | 单位 = {'成交量_股': '股', '成交额_元': '元'}


In [2]:
futures_path = REPO / "src" / "finpass" / "profiles" / "futures.yaml"
assert futures_path.is_file(), "futures.yaml 未落盘"

futures = load_profile("futures")
print("futures key =", futures.keys, "| label =", futures.key_label)
assert len(futures.keys) == 2

limit_rule = futures.rules[-1]
print("期货特有规则：", limit_rule.id, "/", limit_rule.type, "min =", limit_rule.get("min"), "max =", limit_rule.get("max"))
assert limit_rule.type == "range_check"
assert limit_rule.columns == ["涨跌幅_%"]

futures key = ['合约代码', '日期'] | label = 合约代码+日期
期货特有规则： limit_up_down / range_check min = -10.0 max = 10.0


In [3]:
import pandas as pd

futures_csv = REPO / "examples" / "futures_daily.csv"
frame = pd.read_csv(futures_csv)
print("shape =", frame.shape)
print(frame.head(3).to_string(index=False))

# 9 列 = 合约代码/日期/开高低收/成交量_手/持仓量/涨跌幅_%（比原计划多 2 列：
# limit_up_down 需要涨跌幅列，futures 语义下持仓量也是必备列）
assert frame.shape == (10, 9)
assert "合约代码" in frame.columns

shape = (10, 9)
  合约代码         日期  开盘价  最高价  最低价  收盘价  成交量_手   持仓量  涨跌幅_%
IF2406 2024-06-03 3900 3910 3880 3905  12000 45000    0.8
IF2406 2024/06/04 3905 3920 3890 3915  13500 45200    2.1
IF2406 2024-06-05 3915 3930 3900 3925  11000 44800    1.2


In [4]:
import finpass as fp

session = fp.open(futures_csv, profile="futures")
stats = session.start()
print("命中规则：", stats["rules_hit"])
print("问题数：", stats["total_questions"], "| 覆盖行数：", stats["total_affected_rows"])
for question in session.queue:
    print(f"- {question.rule_id:22s} {question.aggregation_level:8s} keys={question.affected_row_keys[:2]}")

assert len(session.queue) >= 1
assert "limit_up_down" in stats["rules_hit"]
# 复合主键：row_key 形态是「合约代码|日期」
assert any("|" in str(key) for key in session.queue[0].affected_row_keys)

命中规则： ['date_format', 'duplicate_key', 'limit_up_down', 'missing_token', 'ohlc_contradiction']
问题数： 5 | 覆盖行数： 5
- duplicate_key          strategy keys=['IC2406|2024-06-05']
- missing_token          column   keys=['IF2409|2024-06-05']
- date_format            column   keys=['IF2406|2024/06/04']
- limit_up_down          column   keys=['IC2406|2024-06-04']
- ohlc_contradiction     point    keys=['IC2406|2024-06-05']


In [5]:
a_stock_csv = REPO / "examples" / "pingan_bank_daily_dirty.csv"

try:
    fp.open(a_stock_csv, profile="futures")
except ValueError as exc:
    print("明确拒绝：", exc)
    assert "缺少必需列" in str(exc)
    assert "合约代码" in str(exc)
else:
    raise AssertionError("用 futures profile 打开 A 股 CSV 竟然成功了：profile.key 形同虚设")

明确拒绝： profile futures_daily 缺少必需列：合约代码、涨跌幅_%


In [6]:
import hashlib

engine_path = REPO / "src" / "finpass" / "engine.py"
baseline = (REPO / "tests" / "baselines" / "engine.sha256").read_text(encoding="ascii").strip()
digest = hashlib.sha256(engine_path.read_bytes()).hexdigest()
print("baseline =", baseline)
print("current  =", digest)
assert digest == baseline, "engine.py 被改动了：新增域只应新增 yaml"

sys.path.insert(0, str(REPO / "tests"))
from _utils import engine_source_without_docstrings

FIN_WORDS = ["收盘", "开盘", "最高价", "最低价", "涨停", "跌停", "涨跌停", "复权", "成交量_股", "成交额_元", "换手率"]
body = engine_source_without_docstrings(engine_path.read_text(encoding="utf-8"))
hits = [word for word in FIN_WORDS if word in body]
print("engine 金融词命中：", hits)
assert hits == []

baseline = 8933beaae0084f20cc1c62970c52988e293007644f9594e8993081b11634a5ce
current  = 8933beaae0084f20cc1c62970c52988e293007644f9594e8993081b11634a5ce
engine 金融词命中： []


In [7]:
print("✅ 04 验收通过 | 新域仅加 yaml | engine hash 不变 | 缺列明确拒绝")

✅ 04 验收通过 | 新域仅加 yaml | engine hash 不变 | 缺列明确拒绝
